In [1]:
# =============================================================
# CELL 0: INSTALL & W&B LOGIN
# =============================================================
import subprocess, sys

subprocess.check_call([sys.executable, "-m", "pip", "install",
                       "transformers", "wandb", "--quiet"])

import os
os.environ["WANDB_PROJECT"] = "24f1001822-t12026"
os.environ["WANDB_SILENT"] = "true"

import wandb
wandb.login(key="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")
print("W&B login successful!")

W&B login successful!


In [2]:
# =============================================================
# CELL 1: IMPORTS
# =============================================================
import gc, random, warnings, time, json
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import librosa
import soundfile as sf

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

try:
    from torch.amp import GradScaler, autocast
    AMP_DEV = "cuda"
except ImportError:
    from torch.cuda.amp import GradScaler, autocast
    AMP_DEV = None

from sklearn.metrics import f1_score, accuracy_score, classification_report

try:
    from transformers import ASTFeatureExtractor, ASTForAudioClassification, ASTConfig
    HAS_TRANSFORMERS = True
    print("transformers loaded OK")
except ImportError:
    HAS_TRANSFORMERS = False
    print("WARNING: transformers not available, AST model will be skipped")

warnings.filterwarnings("ignore")

# --- Reproducibility ---
SEED = 42
def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

seed_everything()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

transformers loaded OK
Device: cuda
GPU: Tesla T4


In [3]:
# =============================================================
# CELL 2: FIND DATASET PATHS
# =============================================================
POSSIBLE_ROOTS = [
    Path("/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup"),
    Path("/kaggle/input/jan-2026-dl-gen-ai-project"),
    Path("/kaggle/input/messy-mashup/messy_mashup"),
    Path("/kaggle/input/messy-mashup"),
]

BASE_DIR = None
for root in POSSIBLE_ROOTS:
    if root.exists() and (root / "genres_stems").exists():
        BASE_DIR = root
        break

if BASE_DIR is None:
    for candidate in Path("/kaggle/input").rglob("genres_stems"):
        BASE_DIR = candidate.parent
        break

if BASE_DIR is None:
    print("ERROR: Cannot find dataset. Contents of /kaggle/input:")
    for item in Path("/kaggle/input").rglob("*"):
        if item.is_dir():
            print(f"  DIR:  {item}")
    raise FileNotFoundError(
        "Cannot find dataset! Make sure you attached the competition dataset. "
        "Click 'Add Input' in the right panel of your Kaggle notebook."
    )

STEMS_DIR   = BASE_DIR / "genres_stems"
MASHUPS_DIR = BASE_DIR / "mashups"
TEST_CSV    = BASE_DIR / "test.csv"
SAMPLE_SUB  = BASE_DIR / "sample_submission.csv"

# Find ESC-50 noise audio
ESC50_DIR = BASE_DIR / "ESC-50-master" / "audio"
if not ESC50_DIR.exists():
    for c in Path("/kaggle/input").rglob("ESC-50*"):
        if c.is_dir():
            ad = c / "audio"
            if ad.exists():
                ESC50_DIR = ad
                break
            if list(c.glob("*.wav")):
                ESC50_DIR = c
                break

print(f"\n--- Dataset Paths ---")
print(f"Base:    {BASE_DIR}")
print(f"Stems:   {STEMS_DIR}  (exists={STEMS_DIR.exists()})")
print(f"Mashups: {MASHUPS_DIR}  (exists={MASHUPS_DIR.exists()})")
print(f"ESC-50:  {ESC50_DIR}  (exists={ESC50_DIR.exists()})")
print(f"Test CSV:{TEST_CSV}  (exists={TEST_CSV.exists()})")

# Genre setup
GENRES = ["blues", "classical", "country", "disco", "hiphop",
          "jazz", "metal", "pop", "reggae", "rock"]
GENRE2IDX = {g: i for i, g in enumerate(GENRES)}
IDX2GENRE = {i: g for g, i in GENRE2IDX.items()}
NUM_CLASSES = len(GENRES)

# Load test data early
test_df = pd.read_csv(TEST_CSV)
sample_sub = pd.read_csv(SAMPLE_SUB)

print(f"\n--- Test Data ---")
print(f"Test samples:     {len(test_df)}")
print(f"Submission cols:  {list(sample_sub.columns)}")
print(f"ID dtype:         {sample_sub['id'].dtype}")
print(f"First 5 IDs:      {sample_sub['id'].head().tolist()}")
print(f"Sample sub head:")
print(sample_sub.head().to_string(index=False))

OUT_PATH = "/kaggle/working/submission.csv"

# Verify genre folders
print(f"\n--- Genre Folders ---")
for g in GENRES:
    gp = STEMS_DIR / g
    print(f"  {g:12s}: {'FOUND' if gp.exists() else 'MISSING'}")


--- Dataset Paths ---
Base:    /kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup
Stems:   /kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems  (exists=True)
Mashups: /kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/mashups  (exists=True)
ESC-50:  /kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/ESC-50-master/audio  (exists=True)
Test CSV:/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/test.csv  (exists=True)

--- Test Data ---
Test samples:     3020
Submission cols:  ['id', 'genre']
ID dtype:         int64
First 5 IDs:      [1, 2, 3, 4, 5]
Sample sub head:
 id     genre
  1      jazz
  2     blues
  3 classical
  4       pop
  5     disco

--- Genre Folders ---
  blues       : FOUND
  classical   : FOUND
  country     : FOUND
  disco       : FOUND
  hiphop      : FOUND
  jazz        : FOUND
  metal       : FOUND
  pop         : FOUND
  reggae      : FOUND
  rock        : FOUND


In [4]:
# =============================================================
# CELL 3: CONFIGURATION
# *** SPEED CHANGES MARKED WITH [SPEED] ***
# =============================================================
class CFG:
    # Audio
    sr = 22050
    duration = 30                   # [FIX] test files are 22-30s long
    n_samples = sr * duration       # 661500

    # Mel spectrogram
    n_mels = 128
    n_fft = 2048
    hop_length = 512
    fmin = 20
    fmax = 8000

    # Training — models were still improving at ep12, need more
    epochs_cnn = 20
    epochs_crnn = 18
    epochs_ast = 8
    batch_size = 16               # [FIX] reduced for 30s spectrograms (3x larger)
    lr_cnn = 3e-4
    lr_crnn = 3e-4
    lr_ast = 5e-5
    weight_decay = 1e-4
    label_smoothing = 0.1
    patience = 6

    # Augmentation [SPEED] removed tempo_prob (time_stretch is very slow)
    mashup_prob = 0.7
    noise_prob = 0.7
    # tempo_prob REMOVED — saves ~3 hours
    noise_snr_range = (5, 25)
    gain_range = (0.4, 1.3)

    # SpecAugment
    specaug_prob = 0.5
    freq_mask_width = 8
    time_mask_width = 15
    n_freq_masks = 2
    n_time_masks = 2

    # Mixup
    mixup_alpha = 0.3
    mixup_prob = 0.4

    # Dataset sizes — increased from v21 (1200 was too few)
    train_samples_per_epoch = 2000
    val_samples = 200

    # Inference [SPEED] reduced TTA
    tta_crops = 3

    # AST
    ast_sr = 16000
    ast_max_length = 1024
    ast_train_samples = 1000
    ast_val_samples = 120


print(f"\nConfig: {CFG.train_samples_per_epoch} train/epoch, "
      f"{CFG.val_samples} val, {CFG.tta_crops} TTA crops")


Config: 2000 train/epoch, 200 val, 3 TTA crops


In [5]:
# =============================================================
# CELL 4: AUDIO UTILITIES
# =============================================================
def load_audio(filepath, sr=CFG.sr, duration=None):
    """Load audio file with librosa."""
    try:
        y, _ = librosa.load(str(filepath), sr=sr, mono=True, duration=duration)
        return y.astype(np.float32)
    except Exception:
        length = int(sr * duration) if duration else sr * 5
        return np.zeros(length, dtype=np.float32)


def pad_or_crop(audio, target_length, random_crop=True):
    """Pad with zeros or crop to exact length."""
    if len(audio) >= target_length:
        if random_crop:
            start = random.randint(0, len(audio) - target_length)
        else:
            start = (len(audio) - target_length) // 2
        return audio[start:start + target_length]
    return np.pad(audio, (0, target_length - len(audio)), mode="constant")


def normalise_audio(audio):
    """Normalise to [-0.95, 0.95]."""
    peak = np.max(np.abs(audio)) + 1e-10
    return (audio / peak * 0.95).astype(np.float32)


def add_noise_at_snr(signal, noise_clip, snr_db):
    """Add noise at given SNR."""
    if len(noise_clip) < len(signal):
        noise_clip = np.tile(noise_clip, (len(signal) // len(noise_clip)) + 1)
    start = random.randint(0, max(0, len(noise_clip) - len(signal)))
    noise_seg = noise_clip[start:start + len(signal)]
    sig_power = np.mean(signal ** 2) + 1e-10
    noise_power = np.mean(noise_seg ** 2) + 1e-10
    scale = np.sqrt(sig_power / (noise_power * (10 ** (snr_db / 10))))
    return signal + scale * noise_seg


def spec_augment(spec, cfg=CFG):
    """SpecAugment: mask random time and frequency bands."""
    s = spec.copy()
    n_mels, n_frames = s.shape
    for _ in range(cfg.n_freq_masks):
        f = random.randint(1, min(cfg.freq_mask_width, n_mels - 1))
        f0 = random.randint(0, n_mels - f)
        s[f0:f0 + f, :] = 0
    for _ in range(cfg.n_time_masks):
        t = random.randint(1, min(cfg.time_mask_width, n_frames - 1))
        t0 = random.randint(0, n_frames - t)
        s[:, t0:t0 + t] = 0
    return s

In [6]:
# =============================================================
# CELL 5: BUILD STEM INDEX + PRE-CACHE AUDIO [SPEED]
# =============================================================
def build_stem_index(stems_dir):
    """Scan genres_stems folder."""
    index = defaultdict(list)
    stem_names = {"drums", "vocals", "bass", "others", "other"}

    for genre in GENRES:
        genre_path = stems_dir / genre
        if not genre_path.exists():
            print(f"  WARNING: {genre} folder missing")
            continue

        subdirs = sorted([d for d in genre_path.iterdir() if d.is_dir()])

        if subdirs:
            for song_dir in subdirs:
                stem_dict = {}
                for wav_file in song_dir.glob("*.wav"):
                    fname = wav_file.stem.lower()
                    for sname in stem_names:
                        if sname in fname:
                            key = "others" if sname == "other" else sname
                            stem_dict[key] = str(wav_file)
                            break
                if stem_dict:
                    index[genre].append(stem_dict)
        else:
            wav_files = sorted(genre_path.glob("*.wav"))
            song_groups = defaultdict(dict)
            for wav_file in wav_files:
                fname = wav_file.stem.lower()
                matched = None
                for sname in stem_names:
                    if sname in fname:
                        matched = "others" if sname == "other" else sname
                        break
                if matched:
                    key_part = fname.split(
                        matched if matched != "others" else "other"
                    )[0].rstrip("._- ")
                    song_groups[key_part][matched] = str(wav_file)
            for _, stem_dict in song_groups.items():
                if stem_dict:
                    index[genre].append(stem_dict)

    return index


print("\n--- Building Stem Index ---")
stem_index = build_stem_index(STEMS_DIR)
total_songs = 0
for g in GENRES:
    count = len(stem_index[g])
    total_songs += count
    stems = set()
    for sd in stem_index[g]:
        stems.update(sd.keys())
    print(f"  {g:12s}: {count:3d} songs | stems: {sorted(stems)}")
print(f"  TOTAL: {total_songs} songs")

# [SPEED] Pre-cache all stem audio in memory
print("\n--- Pre-caching stem audio (avoids repeated disk reads) ---")
STEM_CACHE = {}
cached_count = 0
for genre in GENRES:
    for song in stem_index[genre]:
        for sname, fpath in song.items():
            if fpath not in STEM_CACHE:
                STEM_CACHE[fpath] = load_audio(fpath, CFG.sr)
                cached_count += 1
print(f"  Cached {cached_count} stems in RAM")

# Also pre-cache at AST sample rate
STEM_CACHE_AST = {}
if HAS_TRANSFORMERS:
    print("  Caching stems at 16kHz for AST...")
    for genre in GENRES:
        for song in stem_index[genre]:
            for sname, fpath in song.items():
                if fpath not in STEM_CACHE_AST:
                    STEM_CACHE_AST[fpath] = load_audio(fpath, CFG.ast_sr)
    print(f"  Cached {len(STEM_CACHE_AST)} stems at 16kHz")


def load_cached(fpath, sr=CFG.sr):
    """[SPEED] Load from RAM cache instead of disk."""
    if sr == CFG.ast_sr and fpath in STEM_CACHE_AST:
        return STEM_CACHE_AST[fpath].copy()
    if fpath in STEM_CACHE:
        return STEM_CACHE[fpath].copy()
    return load_audio(fpath, sr)


--- Building Stem Index ---
  blues       : 100 songs | stems: ['bass', 'drums', 'others', 'vocals']
  classical   : 100 songs | stems: ['bass', 'drums', 'others', 'vocals']
  country     : 100 songs | stems: ['bass', 'drums', 'others', 'vocals']
  disco       : 100 songs | stems: ['bass', 'drums', 'others', 'vocals']
  hiphop      : 100 songs | stems: ['bass', 'drums', 'others', 'vocals']
  jazz        : 100 songs | stems: ['bass', 'drums', 'others', 'vocals']
  metal       : 100 songs | stems: ['bass', 'drums', 'others', 'vocals']
  pop         : 100 songs | stems: ['bass', 'drums', 'others', 'vocals']
  reggae      : 100 songs | stems: ['bass', 'drums', 'others', 'vocals']
  rock        : 100 songs | stems: ['bass', 'drums', 'others', 'vocals']
  TOTAL: 1000 songs

--- Pre-caching stem audio (avoids repeated disk reads) ---
  Cached 4000 stems in RAM
  Caching stems at 16kHz for AST...
  Cached 4000 stems at 16kHz


In [7]:
# =============================================================
# CELL 6: LOAD NOISE CLIPS
# =============================================================
def load_noise_clips(esc50_dir, sr=CFG.sr, max_clips=200):  # [SPEED] 200 not 300
    """Load ESC-50 environmental sounds."""
    clips = []
    if not Path(esc50_dir).exists():
        print(f"  ESC-50 not found at {esc50_dir}")
        return clips
    files = list(Path(esc50_dir).glob("*.wav"))
    if not files:
        files = list(Path(esc50_dir).rglob("*.wav"))
    random.shuffle(files)
    for f in files[:max_clips]:
        try:
            y = load_audio(f, sr)
            if len(y) > sr:
                clips.append(y)
        except Exception:
            continue
    print(f"  Loaded {len(clips)} noise clips")
    return clips


print("\n--- Loading Noise Clips ---")
noise_clips = load_noise_clips(ESC50_DIR)

# [SPEED] Also cache noise at AST rate
noise_clips_ast = []
if HAS_TRANSFORMERS and noise_clips:
    print("  Resampling noise to 16kHz for AST...")
    for nc in noise_clips[:100]:
        noise_clips_ast.append(librosa.resample(nc, orig_sr=CFG.sr, target_sr=CFG.ast_sr))
    print(f"  {len(noise_clips_ast)} noise clips at 16kHz")


--- Loading Noise Clips ---
  Loaded 200 noise clips
  Resampling noise to 16kHz for AST...
  100 noise clips at 16kHz


In [8]:
# =============================================================
# CELL 7: MASHUP CREATION
# [SPEED] Removed time_stretch, uses load_cached
# =============================================================
def create_mashup(stem_index, genre, noise_clips, cfg=CFG):
    """Create synthetic mashup from stems."""
    songs = stem_index[genre]
    stem_types = ["drums", "vocals", "bass", "others"]
    n_songs = len(songs)

    if n_songs == 0:
        return np.zeros(cfg.n_samples, dtype=np.float32)

    cross_song = random.random() < cfg.mashup_prob and n_songs >= 2

    if cross_song:
        if n_songs >= 4:
            indices = random.sample(range(n_songs), 4)
        else:
            indices = [random.randint(0, n_songs - 1) for _ in range(4)]
    else:
        idx = random.randint(0, n_songs - 1)
        indices = [idx] * 4

    mixed = np.zeros(cfg.n_samples, dtype=np.float32)
    stems_used = 0

    for i, sname in enumerate(stem_types):
        song = songs[indices[i]]
        if sname in song:
            audio = load_cached(song[sname], cfg.sr)  # [SPEED] cached
        elif song:
            audio = load_cached(list(song.values())[0], cfg.sr)  # [SPEED] cached
        else:
            continue

        if len(audio) < cfg.sr:
            continue

        # [SPEED] time_stretch REMOVED — it was taking ~3 hours total

        audio = pad_or_crop(audio, cfg.n_samples, random_crop=True)
        gain = random.uniform(*cfg.gain_range)
        mixed += gain * audio
        stems_used += 1

    if stems_used == 0:
        return np.zeros(cfg.n_samples, dtype=np.float32)

    mixed = normalise_audio(mixed)

    if random.random() < cfg.noise_prob and noise_clips:
        snr = random.uniform(*cfg.noise_snr_range)
        mixed = add_noise_at_snr(mixed, random.choice(noise_clips), snr)
        if random.random() < 0.25:
            snr2 = random.uniform(15, 30)
            mixed = add_noise_at_snr(mixed, random.choice(noise_clips), snr2)
        mixed = normalise_audio(mixed)

    return mixed

In [9]:
# =============================================================
# CELL 8: FEATURE EXTRACTION
# =============================================================
def audio_to_mel_3ch(audio, cfg=CFG, augment=False):
    """
    3-channel features: mel + delta + delta-delta.
    Returns: (3, 128, T)
    """
    mel = librosa.feature.melspectrogram(
        y=audio, sr=cfg.sr, n_fft=cfg.n_fft,
        hop_length=cfg.hop_length, n_mels=cfg.n_mels,
        fmin=cfg.fmin, fmax=cfg.fmax)
    mel = librosa.power_to_db(mel, ref=np.max)

    if augment and random.random() < cfg.specaug_prob:
        mel = spec_augment(mel, cfg)

    mel_min, mel_max = mel.min(), mel.max()
    if mel_max - mel_min > 1e-6:
        mel_norm = (mel - mel_min) / (mel_max - mel_min)
    else:
        mel_norm = np.zeros_like(mel)

    delta1 = librosa.feature.delta(mel_norm, order=1)
    delta2 = librosa.feature.delta(mel_norm, order=2)

    for arr in [delta1, delta2]:
        mn, mx = arr.min(), arr.max()
        if mx - mn > 1e-6:
            arr[:] = (arr - mn) / (mx - mn)
        else:
            arr[:] = 0.5

    return np.stack([mel_norm, delta1, delta2], axis=0).astype(np.float32)


expected_T = 1 + (CFG.n_samples // CFG.hop_length)
print(f"\nFeature shape: (3, {CFG.n_mels}, {expected_T})")


Feature shape: (3, 128, 1292)


In [10]:
# =============================================================
# CELL 9: DATASETS
# =============================================================
class TrainDataset(Dataset):
    def __init__(self, stem_index, noise_clips, genres, cfg=CFG):
        self.stem_index = stem_index
        self.noise_clips = noise_clips
        self.genres = genres
        self.cfg = cfg

    def __len__(self):
        return self.cfg.train_samples_per_epoch

    def __getitem__(self, idx):
        genre = random.choice(self.genres)
        label = GENRE2IDX[genre]
        audio = create_mashup(self.stem_index, genre, self.noise_clips, self.cfg)
        features = audio_to_mel_3ch(audio, self.cfg, augment=True)
        return torch.from_numpy(features), label


class ValDataset(Dataset):
    def __init__(self, stem_index, noise_clips, cfg=CFG):
        self.data = []
        print(f"  Generating {cfg.val_samples} validation mashups...")
        for i in range(cfg.val_samples):
            genre = GENRES[i % NUM_CLASSES]
            label = GENRE2IDX[genre]
            audio = create_mashup(stem_index, genre, noise_clips, cfg)
            features = audio_to_mel_3ch(audio, cfg, augment=False)
            self.data.append((features, label))
        print(f"  Done.")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        features, label = self.data[idx]
        return torch.from_numpy(features), label


class TestDatasetTTA(Dataset):
    def __init__(self, test_df, mashups_dir, n_crops=3, cfg=CFG):  # [SPEED] default 3
        self.cfg = cfg
        self.n_crops = n_crops
        self.ids = []
        self.audio_cache = []

        mdir = Path(mashups_dir)
        all_files = {f.stem: f for f in mdir.glob("*.wav")}
        print(f"  Test wavs found: {len(all_files)}")

        matched = 0
        for _, row in test_df.iterrows():
            fid = str(row["id"])
            self.ids.append(fid)

            # [FIX] Use the filename column from test.csv to find the file
            fpath = None
            if "filename" in test_df.columns:
                fname = str(row["filename"])
                # filename is like "mashups/song0001.wav" -> stem is "song0001"
                fname_stem = Path(fname).stem
                fpath = all_files.get(fname_stem)

            # Fallback: try matching ID directly
            if fpath is None:
                fpath = all_files.get(fid)

            # Fallback: try songXXXX format
            if fpath is None:
                padded = f"song{int(fid):04d}" if fid.isdigit() else fid
                fpath = all_files.get(padded)

            # Last resort: fuzzy match
            if fpath is None:
                for key, path in all_files.items():
                    if fid in key or key in fid:
                        fpath = path
                        break

            if fpath:
                self.audio_cache.append(load_audio(fpath, cfg.sr))
                matched += 1
            else:
                self.audio_cache.append(np.zeros(cfg.n_samples, dtype=np.float32))
        print(f"  Matched {matched}/{len(test_df)} test files")

    def __len__(self):
        return len(self.ids) * self.n_crops

    def __getitem__(self, idx):
        si = idx // self.n_crops
        ci = idx % self.n_crops
        audio = self.audio_cache[si]
        cropped = pad_or_crop(audio, self.cfg.n_samples, random_crop=(ci > 0))
        features = audio_to_mel_3ch(cropped, self.cfg, augment=False)
        return torch.from_numpy(features), self.ids[si]


class TestDatasetTTA_AST(Dataset):
    def __init__(self, test_df, mashups_dir, feature_extractor, n_crops=3, cfg=CFG):
        self.cfg = cfg
        self.n_crops = n_crops
        self.fe = feature_extractor
        self.ids = []
        self.audio_cache = []

        mdir = Path(mashups_dir)
        all_files = {f.stem: f for f in mdir.glob("*.wav")}

        for _, row in test_df.iterrows():
            fid = str(row["id"])
            self.ids.append(fid)

            # [FIX] Use filename column from test.csv
            fpath = None
            if "filename" in test_df.columns:
                fname_stem = Path(str(row["filename"])).stem
                fpath = all_files.get(fname_stem)
            if fpath is None:
                fpath = all_files.get(fid)
            if fpath is None and fid.isdigit():
                fpath = all_files.get(f"song{int(fid):04d}")
            if fpath is None:
                for key, path in all_files.items():
                    if fid in key or key in fid:
                        fpath = path
                        break

            if fpath:
                self.audio_cache.append(load_audio(fpath, cfg.ast_sr))
            else:
                self.audio_cache.append(
                    np.zeros(int(cfg.ast_sr * cfg.duration), dtype=np.float32))

    def __len__(self):
        return len(self.ids) * self.n_crops

    def __getitem__(self, idx):
        si = idx // self.n_crops
        ci = idx % self.n_crops
        audio = self.audio_cache[si]
        target = int(self.cfg.ast_sr * self.cfg.duration)
        cropped = pad_or_crop(audio, target, random_crop=(ci > 0))
        inputs = self.fe(cropped, sampling_rate=self.cfg.ast_sr, return_tensors="pt")
        return inputs["input_values"].squeeze(0), self.ids[si]

In [11]:
# =============================================================
# CELL 10: MODEL 1 — CNN FROM SCRATCH
# =============================================================
class ResBlock(nn.Module):
    """Residual block with skip connection."""
    def __init__(self, channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels))
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.relu(x + self.block(x))


class GenreCNN(nn.Module):
    """
    MODEL 1: Custom CNN built from scratch.
    ResNet-style architecture with residual connections.
    Input: (batch, 3, 128, T) — mel + delta + delta-delta
    Output: (batch, 10) — genre logits
    """
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1))

        self.layer1 = nn.Sequential(
            nn.Conv2d(64, 128, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            ResBlock(128), nn.Dropout2d(0.1))

        self.layer2 = nn.Sequential(
            nn.Conv2d(128, 256, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            ResBlock(256), nn.Dropout2d(0.15))

        self.layer3 = nn.Sequential(
            nn.Conv2d(256, 512, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(512), nn.ReLU(inplace=True),
            ResBlock(512), nn.Dropout2d(0.2))

        self.gap = nn.AdaptiveAvgPool2d(1)
        self.gmp = nn.AdaptiveMaxPool2d(1)

        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(1024, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes))

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        avg_pool = self.gap(x)
        max_pool = self.gmp(x)
        x = torch.cat([avg_pool, max_pool], dim=1)
        return self.head(x)

In [12]:
# =============================================================
# CELL 11: MODEL 2 — CRNN (CNN + GRU)
# =============================================================
class GenreCRNN(nn.Module):
    """
    MODEL 2: CNN + bidirectional GRU.
    CNN extracts spectral features, GRU captures temporal patterns.
    Built from scratch.
    """
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(3, 64, 3, stride=(2, 1), padding=1, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.MaxPool2d((2, 2)), nn.Dropout2d(0.1),

            nn.Conv2d(64, 128, 3, stride=(2, 1), padding=1, bias=False),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, 3, padding=1, bias=False),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.MaxPool2d((2, 2)), nn.Dropout2d(0.15),

            nn.Conv2d(128, 256, 3, stride=(2, 1), padding=1, bias=False),
            nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.MaxPool2d((2, 2)), nn.Dropout2d(0.2))

        self.gru = nn.GRU(
            input_size=256 * 2,
            hidden_size=128,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=0.2)

        self.head = nn.Sequential(
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(128, num_classes))

    def forward(self, x):
        x = self.cnn(x)
        B, C, F, T = x.shape
        x = x.permute(0, 3, 1, 2).reshape(B, T, C * F)
        out, _ = self.gru(x)
        x = out[:, -1, :]
        return self.head(x)

In [13]:
# =============================================================
# CELL 12: MODEL 3 — AST (Pretrained)
# =============================================================
if HAS_TRANSFORMERS:
    class GenreAST(nn.Module):
        """
        MODEL 3: Audio Spectrogram Transformer.
        Pretrained on AudioSet, fine-tuned for 10 music genres.
        """
        def __init__(self, num_classes=NUM_CLASSES):
            super().__init__()
            config = ASTConfig.from_pretrained(
                "MIT/ast-finetuned-audioset-10-10-0.4593")
            config.num_labels = num_classes
            self.ast = ASTForAudioClassification.from_pretrained(
                "MIT/ast-finetuned-audioset-10-10-0.4593",
                config=config,
                ignore_mismatched_sizes=True)

        def forward(self, x):
            return self.ast(x).logits

In [14]:
# =============================================================
# CELL 13: TRAINING ENGINE
# =============================================================
def do_autocast():
    if AMP_DEV is not None:
        return autocast(AMP_DEV)
    return autocast()


def mixup_data(x, y, alpha=0.3):
    lam = np.random.beta(alpha, alpha)
    index = torch.randperm(x.size(0)).to(x.device)
    mixed_x = lam * x + (1 - lam) * x[index]
    return mixed_x, y, y[index], lam


class EarlyStopping:
    def __init__(self, patience=5, min_delta=0.001):  # [SPEED] patience 5
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_score = None
        self.should_stop = False

    def __call__(self, score):
        if self.best_score is None or score > self.best_score + self.min_delta:
            self.best_score = score
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True


def train_model(model, train_loader, val_loader, epochs, lr, model_name, cfg=CFG):
    """Train model with W&B logging."""
    print(f"\n  Training: {model_name} | Epochs: {epochs} | LR: {lr}")

    wandb.init(
        project=os.environ.get("WANDB_PROJECT", "24f1001822-t12026"),
        name=model_name,
        config={
            "model": model_name, "epochs": epochs, "lr": lr,
            "batch_size": cfg.batch_size, "label_smoothing": cfg.label_smoothing,
            "mixup_alpha": cfg.mixup_alpha, "mixup_prob": cfg.mixup_prob,
            "mashup_prob": cfg.mashup_prob, "noise_prob": cfg.noise_prob,
            "train_samples": cfg.train_samples_per_epoch,
            "val_samples": cfg.val_samples},
        reinit=True)

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=lr, weight_decay=cfg.weight_decay)
    total_steps = len(train_loader) * epochs
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=lr, total_steps=total_steps,
        pct_start=0.15, anneal_strategy="cos")
    scaler = GradScaler()
    criterion = nn.CrossEntropyLoss(label_smoothing=cfg.label_smoothing)
    early_stop = EarlyStopping(patience=cfg.patience)

    best_f1 = 0
    best_state = None

    for epoch in range(epochs):
        model.train()
        total_loss, n_batches = 0, 0
        t0 = time.time()

        for features, labels in train_loader:
            features = features.to(DEVICE)
            if isinstance(labels, torch.Tensor):
                labels = labels.long().to(DEVICE)
            else:
                labels = torch.tensor(labels).long().to(DEVICE)

            use_mixup = random.random() < cfg.mixup_prob
            if use_mixup:
                features, ya, yb, lam = mixup_data(features, labels, cfg.mixup_alpha)

            optimizer.zero_grad()
            with do_autocast():
                logits = model(features)
                if use_mixup:
                    loss = lam * criterion(logits, ya) + (1 - lam) * criterion(logits, yb)
                else:
                    loss = criterion(logits, labels)

            scaler.scale(loss).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            total_loss += loss.item()
            n_batches += 1

        avg_loss = total_loss / max(n_batches, 1)

        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for features, labels in val_loader:
                features = features.to(DEVICE)
                with do_autocast():
                    logits = model(features)
                all_preds.extend(logits.argmax(-1).cpu().tolist())
                if isinstance(labels, torch.Tensor):
                    all_labels.extend(labels.tolist())
                else:
                    all_labels.extend(list(labels))

        val_f1 = f1_score(all_labels, all_preds, average="macro")
        val_acc = accuracy_score(all_labels, all_preds)
        elapsed = time.time() - t0

        print(f"  [{model_name}] Ep {epoch+1:2d}/{epochs} | "
              f"Loss {avg_loss:.4f} | Val F1 {val_f1:.4f} | Acc {val_acc:.4f} | {elapsed:.0f}s")

        wandb.log({
            "epoch": epoch + 1, "train_loss": avg_loss,
            "val_macro_f1": val_f1,
            "val_accuracy": val_acc,
            "learning_rate": optimizer.param_groups[0]["lr"]})

        if val_f1 > best_f1:
            best_f1 = val_f1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        early_stop(val_f1)
        if early_stop.should_stop:
            print(f"  [{model_name}] Early stopping at epoch {epoch+1}")
            break

    print(f"  [{model_name}] Best Val F1: {best_f1:.4f}")

    # Print per-class classification report (useful for report and viva)
    if all_preds and all_labels:
        print(f"\n  [{model_name}] Classification Report (last epoch):")
        print(classification_report(all_labels, all_preds,
              target_names=GENRES, digits=3))

    wandb.log({"best_val_macro_f1": best_f1, "best_val_accuracy": val_acc})
    wandb.finish()

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_f1

In [15]:
# =============================================================
# CELL 14: INFERENCE WITH TTA
# =============================================================
@torch.no_grad()
def predict_with_tta(model, test_df, mashups_dir, cfg=CFG,
                     is_ast=False, feature_extractor=None):
    """Predict with test-time augmentation."""
    model.eval()
    if is_ast and feature_extractor is not None:
        dataset = TestDatasetTTA_AST(
            test_df, mashups_dir, feature_extractor, n_crops=3, cfg=cfg)
    else:
        dataset = TestDatasetTTA(
            test_df, mashups_dir, n_crops=cfg.tta_crops, cfg=cfg)

    loader = DataLoader(dataset, batch_size=cfg.batch_size,
                        shuffle=False, num_workers=2, pin_memory=True)

    all_logits = defaultdict(list)
    for features, file_ids in loader:
        features = features.to(DEVICE)
        with do_autocast():
            logits = model(features)
        logits_np = logits.cpu().numpy()
        for i, fid in enumerate(file_ids):
            all_logits[fid].append(logits_np[i])

    predictions = {}
    for fid, logit_list in all_logits.items():
        predictions[fid] = np.mean(logit_list, axis=0)
    return predictions

In [16]:
# =============================================================
# CELL 15: SUBMISSION WRITER
# =============================================================
def write_submission(predictions_dict, sample_sub, output_path):
    """Write submission matching sample_submission.csv exactly."""
    submission = sample_sub.copy()
    str_preds = {str(k): v for k, v in predictions_dict.items()}

    def get_genre(row_id):
        sid = str(row_id)
        if sid in str_preds:
            val = str_preds[sid]
            if isinstance(val, np.ndarray):
                return IDX2GENRE[np.argmax(val)]
            return val
        return "rock"

    submission["genre"] = submission["id"].apply(get_genre)

    valid = set(GENRES)
    bad_mask = ~submission["genre"].isin(valid)
    if bad_mask.any():
        print(f"  WARNING: {bad_mask.sum()} invalid genres, setting to 'rock'")
        submission.loc[bad_mask, "genre"] = "rock"

    submission.to_csv(output_path, index=False)
    print(f"\n  Saved: {output_path} | Shape: {submission.shape}")
    print(f"  Distribution:\n  {submission['genre'].value_counts().to_string()}")

    assert len(submission) == len(sample_sub), "Row count mismatch!"
    assert list(submission.columns) == list(sample_sub.columns), "Column mismatch!"
    assert submission["genre"].isin(valid).all(), "Invalid genres!"
    print("  ALL CHECKS PASSED")
    return submission

In [17]:
# =============================================================
# CELL 16: SAFETY NET
# =============================================================
print("\n" + "=" * 60)
print("  SAFETY NET: Writing baseline submission")
print("=" * 60)
baseline = {str(row["id"]): random.choice(GENRES) for _, row in test_df.iterrows()}
write_submission(baseline, sample_sub, OUT_PATH)
print("  Baseline ready. Training will overwrite.\n")


  SAFETY NET: Writing baseline submission

  Saved: /kaggle/working/submission.csv | Shape: (3020, 2)
  Distribution:
  genre
jazz         315
reggae       311
country      309
classical    309
metal        307
rock         301
pop          301
hiphop       293
blues        288
disco        286
  ALL CHECKS PASSED
  Baseline ready. Training will overwrite.



In [18]:
# =============================================================
# CELL 17: TRAIN ALL 3 MODELS
# =============================================================
try:
    available_genres = [g for g in GENRES if len(stem_index[g]) > 0]
    print(f"\nGenres with data: {len(available_genres)}/{NUM_CLASSES}")

    all_model_predictions = []
    model_names_used = []
    model_f1_scores = {}

    # =========================================================
    # MODEL 1: CNN FROM SCRATCH
    # =========================================================
    print(f"\n{'='*60}")
    print("  MODEL 1: CNN (from scratch)")
    print(f"{'='*60}")

    seed_everything(SEED)
    model1 = GenreCNN(NUM_CLASSES).to(DEVICE)
    print(f"  Parameters: {sum(p.numel() for p in model1.parameters()):,}")

    train_ds = TrainDataset(stem_index, noise_clips, available_genres, CFG)
    val_ds = ValDataset(stem_index, noise_clips, CFG)
    train_loader = DataLoader(
        train_ds, batch_size=CFG.batch_size, shuffle=True,
        num_workers=2, pin_memory=True, drop_last=True)
    val_loader = DataLoader(
        val_ds, batch_size=CFG.batch_size * 2, shuffle=False,
        num_workers=2, pin_memory=True)

    model1, f1_cnn = train_model(
        model1, train_loader, val_loader,
        CFG.epochs_cnn, CFG.lr_cnn, "Model1_CNN_scratch", CFG)
    model_f1_scores["CNN"] = f1_cnn

    print(f"\n  Predicting with CNN...")
    preds1 = predict_with_tta(model1, test_df, MASHUPS_DIR, CFG)
    all_model_predictions.append(preds1)
    model_names_used.append("CNN")

    del model1; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

    # =========================================================
    # MODEL 2: CRNN (CNN + GRU)
    # =========================================================
    print(f"\n{'='*60}")
    print("  MODEL 2: CRNN (CNN + GRU)")
    print(f"{'='*60}")

    seed_everything(SEED + 100)
    model2 = GenreCRNN(NUM_CLASSES).to(DEVICE)
    print(f"  Parameters: {sum(p.numel() for p in model2.parameters()):,}")

    train_ds2 = TrainDataset(stem_index, noise_clips, available_genres, CFG)
    val_ds2 = ValDataset(stem_index, noise_clips, CFG)
    train_loader2 = DataLoader(
        train_ds2, batch_size=CFG.batch_size, shuffle=True,
        num_workers=2, pin_memory=True, drop_last=True)
    val_loader2 = DataLoader(
        val_ds2, batch_size=CFG.batch_size * 2, shuffle=False,
        num_workers=2, pin_memory=True)

    model2, f1_crnn = train_model(
        model2, train_loader2, val_loader2,
        CFG.epochs_crnn, CFG.lr_crnn, "Model2_CRNN_GRU", CFG)
    model_f1_scores["CRNN"] = f1_crnn

    print(f"\n  Predicting with CRNN...")
    preds2 = predict_with_tta(model2, test_df, MASHUPS_DIR, CFG)
    all_model_predictions.append(preds2)
    model_names_used.append("CRNN")

    del model2, train_ds2, val_ds2, train_loader2, val_loader2
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

    # =========================================================
    # MODEL 3: AST (Pretrained)
    # =========================================================
    if HAS_TRANSFORMERS:
        print(f"\n{'='*60}")
        print("  MODEL 3: AST (pretrained transformer)")
        print(f"{'='*60}")

        seed_everything(SEED + 200)

        ast_fe = ASTFeatureExtractor.from_pretrained(
            "MIT/ast-finetuned-audioset-10-10-0.4593")
        print(f"  AST sampling rate: {ast_fe.sampling_rate}")

        class TrainDataset_AST(Dataset):
            def __init__(self, stem_index, noise_clips, genres, fe, cfg=CFG):
                self.stem_index = stem_index
                self.noise_clips = noise_clips
                self.genres = genres
                self.fe = fe
                self.cfg = cfg

            def __len__(self):
                return self.cfg.ast_train_samples  # [SPEED] fewer

            def __getitem__(self, idx):
                genre = random.choice(self.genres)
                label = GENRE2IDX[genre]
                # Create mashup at AST sample rate
                import copy
                ast_cfg = copy.copy(self.cfg)
                ast_cfg.sr = self.cfg.ast_sr
                ast_cfg.n_samples = int(self.cfg.ast_sr * self.cfg.duration)
                audio = create_mashup(self.stem_index, genre,
                                      noise_clips_ast if noise_clips_ast else self.noise_clips,
                                      ast_cfg)
                inputs = self.fe(audio, sampling_rate=self.cfg.ast_sr, return_tensors="pt")
                return inputs["input_values"].squeeze(0), label

        class ValDataset_AST(Dataset):
            def __init__(self, stem_index, noise_clips, fe, cfg=CFG):
                self.data = []
                import copy
                ast_cfg = copy.copy(cfg)
                ast_cfg.sr = cfg.ast_sr
                ast_cfg.n_samples = int(cfg.ast_sr * cfg.duration)
                print(f"  Generating {cfg.ast_val_samples} AST validation samples...")
                for i in range(cfg.ast_val_samples):  # [SPEED] fewer
                    genre = GENRES[i % NUM_CLASSES]
                    label = GENRE2IDX[genre]
                    audio = create_mashup(stem_index, genre,
                                          noise_clips_ast if noise_clips_ast else noise_clips,
                                          ast_cfg)
                    inputs = fe(audio, sampling_rate=cfg.ast_sr, return_tensors="pt")
                    self.data.append((inputs["input_values"].squeeze(0), label))

            def __len__(self):
                return len(self.data)

            def __getitem__(self, idx):
                return self.data[idx]

        try:
            model3 = GenreAST(NUM_CLASSES).to(DEVICE)
            print(f"  Total params: {sum(p.numel() for p in model3.parameters()):,}")

            # Freeze first 8 of 12 encoder layers
            for name, param in model3.ast.named_parameters():
                if "encoder.layer" in name:
                    layer_num = int(name.split("encoder.layer.")[1].split(".")[0])
                    if layer_num < 8:
                        param.requires_grad = False

            trainable = sum(p.numel() for p in model3.parameters() if p.requires_grad)
            print(f"  Trainable params: {trainable:,}")

            ast_bs = min(CFG.batch_size, 16)
            train_ds3 = TrainDataset_AST(stem_index, noise_clips, available_genres, ast_fe, CFG)
            val_ds3 = ValDataset_AST(stem_index, noise_clips, ast_fe, CFG)
            train_loader3 = DataLoader(
                train_ds3, batch_size=ast_bs, shuffle=True,
                num_workers=2, pin_memory=True, drop_last=True)
            val_loader3 = DataLoader(
                val_ds3, batch_size=ast_bs, shuffle=False,
                num_workers=2, pin_memory=True)

            model3, f1_ast = train_model(
                model3, train_loader3, val_loader3,
                CFG.epochs_ast, CFG.lr_ast, "Model3_AST_pretrained", CFG)
            model_f1_scores["AST"] = f1_ast

            print(f"\n  Predicting with AST...")
            preds3 = predict_with_tta(
                model3, test_df, MASHUPS_DIR, CFG,
                is_ast=True, feature_extractor=ast_fe)
            all_model_predictions.append(preds3)
            model_names_used.append("AST")

            del model3, train_ds3, val_ds3, train_loader3, val_loader3
            gc.collect()
            if torch.cuda.is_available(): torch.cuda.empty_cache()

        except Exception as e:
            print(f"\n  AST FAILED: {e}")
            import traceback
            traceback.print_exc()
    else:
        print("\n  SKIPPING AST (transformers not available)")

    # =========================================================
    # ENSEMBLE
    # =========================================================
    print(f"\n{'='*60}")
    print(f"  ENSEMBLING {len(all_model_predictions)} MODELS: {model_names_used}")
    print(f"{'='*60}")

    print(f"\n  Model F1 scores:")
    for name, score in model_f1_scores.items():
        print(f"    {name}: {score:.4f}")

    final_predictions = {}
    for _, row in test_df.iterrows():
        fid = str(row["id"])
        logits_list = []
        for model_preds in all_model_predictions:
            if fid in model_preds:
                logits = model_preds[fid]
                probs = np.exp(logits - np.max(logits))
                probs = probs / probs.sum()
                logits_list.append(probs)
        if logits_list:
            final_predictions[fid] = np.mean(logits_list, axis=0)
        else:
            final_predictions[fid] = "rock"

    # =========================================================
    # WRITE FINAL SUBMISSION
    # =========================================================
    print(f"\n{'='*60}")
    print("  WRITING FINAL SUBMISSION")
    print(f"{'='*60}")
    write_submission(final_predictions, sample_sub, OUT_PATH)

    print(f"\n{'='*60}")
    print("  TRAINING COMPLETE")
    print(f"  Models: {model_names_used}")
    print(f"  Submission: {OUT_PATH}")
    print(f"{'='*60}")

except Exception as e:
    print(f"\n{'!'*60}")
    print(f"  TRAINING FAILED: {e}")
    import traceback
    traceback.print_exc()
    print(f"\n  Baseline exists at {OUT_PATH}")
    print(f"{'!'*60}")


Genres with data: 10/10

  MODEL 1: CNN (from scratch)
  Parameters: 8,021,834
  Generating 200 validation mashups...
  Done.

  Training: Model1_CNN_scratch | Epochs: 20 | LR: 0.0003
  [Model1_CNN_scratch] Ep  1/20 | Loss 2.4162 | Val F1 0.1075 | Acc 0.2050 | 130s
  [Model1_CNN_scratch] Ep  2/20 | Loss 2.2290 | Val F1 0.1023 | Acc 0.1650 | 131s
  [Model1_CNN_scratch] Ep  3/20 | Loss 2.0497 | Val F1 0.3550 | Acc 0.4150 | 132s
  [Model1_CNN_scratch] Ep  4/20 | Loss 1.8651 | Val F1 0.2161 | Acc 0.2700 | 130s
  [Model1_CNN_scratch] Ep  5/20 | Loss 1.7561 | Val F1 0.5123 | Acc 0.5350 | 131s
  [Model1_CNN_scratch] Ep  6/20 | Loss 1.7038 | Val F1 0.4646 | Acc 0.5000 | 131s
  [Model1_CNN_scratch] Ep  7/20 | Loss 1.5889 | Val F1 0.5772 | Acc 0.5850 | 132s
  [Model1_CNN_scratch] Ep  8/20 | Loss 1.5834 | Val F1 0.4849 | Acc 0.5000 | 132s
  [Model1_CNN_scratch] Ep  9/20 | Loss 1.5385 | Val F1 0.6200 | Acc 0.6350 | 131s
  [Model1_CNN_scratch] Ep 10/20 | Loss 1.4406 | Val F1 0.7514 | Acc 0.7500 | 

preprocessor_config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

  AST sampling rate: 16000


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

ASTForAudioClassification LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                     | Status   |                                                                                          
------------------------+----------+------------------------------------------------------------------------------------------
classifier.dense.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([527, 768]) vs model:torch.Size([10, 768])
classifier.dense.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([527]) vs model:torch.Size([10])          

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


  Total params: 86,196,490
  Trainable params: 29,493,514
  Generating 120 AST validation samples...

  Training: Model3_AST_pretrained | Epochs: 8 | LR: 5e-05
  [Model3_AST_pretrained] Ep  1/8 | Loss 2.1206 | Val F1 0.6202 | Acc 0.6500 | 69s
  [Model3_AST_pretrained] Ep  2/8 | Loss 1.2362 | Val F1 0.8058 | Acc 0.8083 | 68s
  [Model3_AST_pretrained] Ep  3/8 | Loss 1.0039 | Val F1 0.8425 | Acc 0.8500 | 67s
  [Model3_AST_pretrained] Ep  4/8 | Loss 1.1327 | Val F1 0.8802 | Acc 0.8833 | 67s
  [Model3_AST_pretrained] Ep  5/8 | Loss 0.9925 | Val F1 0.9249 | Acc 0.9250 | 67s
  [Model3_AST_pretrained] Ep  6/8 | Loss 0.8709 | Val F1 0.9247 | Acc 0.9250 | 67s
  [Model3_AST_pretrained] Ep  7/8 | Loss 0.8938 | Val F1 0.9159 | Acc 0.9167 | 67s
  [Model3_AST_pretrained] Ep  8/8 | Loss 0.9123 | Val F1 0.9159 | Acc 0.9167 | 67s
  [Model3_AST_pretrained] Best Val F1: 0.9249

  [Model3_AST_pretrained] Classification Report (last epoch):
              precision    recall  f1-score   support

       blues

In [19]:
# =============================================================
# CELL 18: FINAL VERIFICATION
# =============================================================
print(f"\n{'='*60}")
print("  FINAL VERIFICATION")
print(f"{'='*60}")

assert os.path.exists(OUT_PATH), f"CRITICAL: {OUT_PATH} missing!"
final_check = pd.read_csv(OUT_PATH)
print(f"  File:       {OUT_PATH}")
print(f"  Rows:       {len(final_check)}")
print(f"  Columns:    {list(final_check.columns)}")
print(f"  ID dtype:   {final_check['id'].dtype}")
print(f"  Valid:       {final_check['genre'].isin(set(GENRES)).all()}")
print(f"\n  Head:")
print(final_check.head(10).to_string(index=False))
print(f"\n  Distribution:")
print(final_check["genre"].value_counts().to_string())
print(f"\n  READY TO SUBMIT")


  FINAL VERIFICATION
  File:       /kaggle/working/submission.csv
  Rows:       3020
  Columns:    ['id', 'genre']
  ID dtype:   int64
  Valid:       True

  Head:
 id     genre
  1       pop
  2 classical
  3     disco
  4     metal
  5   country
  6       pop
  7      rock
  8       pop
  9       pop
 10     disco

  Distribution:
genre
blues        422
hiphop       369
disco        324
rock         321
metal        320
pop          312
reggae       309
jazz         291
classical    223
country      129

  READY TO SUBMIT
